# 03 — Feature Engineering & Dataset Design

**Project:** `ecommerce-delivery-delay-prediction`  
**Phase:** 3 — Feature Engineering & Dataset Design

## Goals

This notebook converts the findings from Phase 2 into a reproducible order-level feature dataset.

The invariant is:

**1 row = 1 order**

The prediction point remains:

`order_approved_at`

Only information available at or before that timestamp may enter the model feature matrix.

This notebook will:

- build reusable order-level features;
- aggregate all 1:N relational tables before joining;
- validate the feature contract and leakage rules;
- inspect missingness;
- create chronological train/validation/test splits;
- define the preprocessing contract;
- export processed datasets for Phase 4.

## 1. Imports

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 2. Project paths

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate

    raise FileNotFoundError(
        "Không tìm thấy project root chứa data/raw."
    )


PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
METRICS_DIR = PROJECT_ROOT / "reports" / "metrics"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_ROOT: /home/namdp/Documents/Projects/ecommerce-delivery-delay-prediction
PROCESSED_DIR: /home/namdp/Documents/Projects/ecommerce-delivery-delay-prediction/data/processed


## 3. Import reusable feature code

The feature logic lives in `src/features/build_features.py`.

The notebook is used for inspection and validation; the Python module is the reusable production-oriented implementation.

In [3]:
from src.features.build_features import (
    CATEGORICAL_FEATURES,
    FORBIDDEN_MODEL_FEATURES,
    MODEL_FEATURE_COLUMNS,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    build_feature_dataset,
    load_project_tables,
    make_chronological_splits,
    validate_feature_dataset,
)

## 4. Load cleaned population and raw relational tables

In [4]:
tables = load_project_tables(PROJECT_ROOT)

orders_cleaned = tables["orders_cleaned"]

print("Cleaned modeling population:", orders_cleaned.shape)

assert len(orders_cleaned) == 96_450
assert orders_cleaned["order_id"].is_unique

display(
    orders_cleaned[
        [
            "order_id",
            "order_approved_at",
            "late_delivery",
        ]
    ].head()
)

Cleaned modeling population: (96450, 10)


,order_id,order_approved_at,late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 11:07:15,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-26 03:24:27,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:55:23,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:45:59,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 22:20:29,0


## 5. Build the feature dataset

All one-to-many raw tables are aggregated independently to order level before being merged.

The output contains only:

- `order_id`;
- `prediction_timestamp`;
- safe model features;
- `late_delivery`.

In [5]:
features = build_feature_dataset(**tables)

print("Feature dataset shape:", features.shape)

display(features.head())

Feature dataset shape: (96450, 44)


,order_id,prediction_timestamp,purchase_year,purchase_month,purchase_weekday,purchase_hour,purchase_is_weekend,purchase_month_sin,purchase_month_cos,purchase_weekday_sin,purchase_weekday_cos,purchase_hour_sin,purchase_hour_cos,approval_lag_hours,promised_delivery_days,item_count,unique_products,seller_count,total_price,total_freight,order_total_value,mean_item_price,max_item_price,freight_ratio,payment_records,payment_value,max_installments,payment_type_count,mean_product_weight_g,total_product_weight_g,mean_product_volume_cm3,total_product_volume_cm3,category_count,mean_product_photos,seller_state_count,mean_distance_km,max_distance_km,min_distance_km,all_sellers_same_state,same_state_seller_share,customer_state,primary_payment_type,dominant_product_category,late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 11:07:15,2017,10,0,10,0,-1.0000,-0.0000,0.0000,1.0000,0.5000,-0.8660,0.1783,15.5366,1,1,1,29.9900,8.7200,38.7100,29.9900,29.9900,0.2253,3.0000,38.7100,1.0000,2.0000,500.0000,500.0000,"1,976.0000","1,976.0000",1,4.0000,1,18.6575,18.6575,18.6575,1,1.0000,SP,credit_card,housewares,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-26 03:24:27,2018,7,1,20,0,0.0000,-1.0000,0.7818,0.6235,-0.8660,0.5000,30.7139,17.8580,1,1,1,118.7000,22.7600,141.4600,118.7000,118.7000,0.1609,1.0000,141.4600,1.0000,1.0000,400.0000,400.0000,"4,693.0000","4,693.0000",1,1.0000,1,861.0699,861.0699,861.0699,0,0.0000,BA,boleto,perfumery,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:55:23,2018,8,2,8,0,-0.5000,-0.8660,0.9749,-0.2225,0.8660,-0.5000,0.2761,26.6282,1,1,1,159.9000,19.2200,179.1200,159.9000,159.9000,0.1073,1.0000,179.1200,3.0000,1.0000,420.0000,420.0000,"9,576.0000","9,576.0000",1,1.0000,1,514.5614,514.5614,514.5614,0,0.0000,GO,credit_card,auto,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:45:59,2017,11,5,19,1,-0.8660,0.5000,-0.9749,-0.2225,-0.9659,0.2588,0.2981,26.1764,1,1,1,45.0000,27.2000,72.2000,45.0000,45.0000,0.3767,1.0000,72.2000,1.0000,1.0000,450.0000,450.0000,"6,000.0000","6,000.0000",1,3.0000,1,"1,821.8742","1,821.8742","1,821.8742",0,0.0000,RN,credit_card,pet_shop,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 22:20:29,2018,2,1,21,0,0.5000,0.8660,0.7818,0.6235,-0.7071,0.7071,1.0306,12.0691,1,1,1,19.9000,8.7200,28.6200,19.9000,19.9000,0.3047,1.0000,28.6200,1.0000,1.0000,250.0000,250.0000,"11,475.0000","11,475.0000",1,4.0000,1,29.6239,29.6239,29.6239,1,1.0000,SP,credit_card,stationery,0


## 6. Validate modeling grain and feature contract

In [6]:
validate_feature_dataset(
    features,
    expected_order_count=len(orders_cleaned),
)

assert features["order_id"].is_unique
assert len(features) == len(orders_cleaned)

print(
    f"✓ Modeling grain preserved: "
    f"{len(features):,} unique orders."
)

✓ Modeling grain preserved: 96,450 unique orders.


### 6.1 Leakage audit

In [7]:
registered_forbidden = (
    FORBIDDEN_MODEL_FEATURES
    & set(MODEL_FEATURE_COLUMNS)
)

processed_forbidden = (
    FORBIDDEN_MODEL_FEATURES
    & set(features.columns)
)

print(
    "Forbidden features registered in X:",
    sorted(registered_forbidden),
)

print(
    "Forbidden columns present in processed dataset:",
    sorted(processed_forbidden),
)

assert not registered_forbidden
assert not processed_forbidden

print("✓ Leakage audit passed.")

Forbidden features registered in X: []
Forbidden columns present in processed dataset: []
✓ Leakage audit passed.


## 7. Feature contract

In [8]:
feature_contract = pd.DataFrame(
    {
        "feature": MODEL_FEATURE_COLUMNS,
        "type": [
            (
                "numeric"
                if feature in NUMERIC_FEATURES
                else "categorical"
            )
            for feature in MODEL_FEATURE_COLUMNS
        ],
    }
)

display(feature_contract)

print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))
print("Total model features:", len(MODEL_FEATURE_COLUMNS))

,feature,type
0,purchase_year,numeric
1,purchase_month,numeric
2,purchase_weekday,numeric
3,purchase_hour,numeric
4,purchase_is_weekend,numeric
5,purchase_month_sin,numeric
6,purchase_month_cos,numeric
7,purchase_weekday_sin,numeric
8,purchase_weekday_cos,numeric
9,purchase_hour_sin,numeric


Numeric features: 38
Categorical features: 3
Total model features: 41


## 8. Important safe derived features

Key prediction-time features include:

- `approval_lag_hours`: time from purchase to approval;
- `promised_delivery_days`: estimated delivery minus approval;
- temporal context from purchase timestamp;
- order value, freight and item aggregates;
- product weight, volume and category aggregates;
- payment aggregates;
- customer state;
- seller–customer distance;
- same-state fulfillment.

`promised_delivery_days` must never be negative after Phase 2 cleaning.

In [9]:
display(
    features[
        [
            "approval_lag_hours",
            "promised_delivery_days",
        ]
    ].describe()
)

assert (
    features["promised_delivery_days"]
    >= 0
).all()

print("✓ All promised delivery windows are valid.")

,approval_lag_hours,promised_delivery_days
count,"96,450.0000","96,450.0000"
mean,10.2666,23.3081
std,20.4805,8.7628
min,0.0000,0.0205
25%,0.2153,17.9062
50%,0.3433,22.6700
75%,14.5128,28.0676
max,741.4436,153.5760


✓ All promised delivery windows are valid.


## 9. Missingness

In [10]:
missing_summary = (
    features[MODEL_FEATURE_COLUMNS]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_summary["missing_pct"] = (
    missing_summary["missing_count"]
    / len(features)
    * 100
)

missing_summary = missing_summary.sort_values(
    ["missing_pct", "missing_count"],
    ascending=False,
)

display(
    missing_summary[
        missing_summary["missing_count"] > 0
    ]
)

,missing_count,missing_pct
mean_product_photos,1331,1.3800
mean_distance_km,476,0.4935
max_distance_km,476,0.4935
min_distance_km,476,0.4935
mean_product_weight_g,16,0.0166
mean_product_volume_cm3,16,0.0166
payment_records,1,0.0010
payment_value,1,0.0010
max_installments,1,0.0010
payment_type_count,1,0.0010


### Missing-value decision

Do **not** drop otherwise valid orders simply because an aggregated feature is missing.

Missing values will be handled inside the sklearn preprocessing pipeline:

- numeric → median imputation;
- categorical → most-frequent imputation;
- categorical encoding → OneHotEncoder with `handle_unknown="ignore"`.

The preprocessing pipeline must be fitted on the training split only.

## 10. Target consistency

In [11]:
target_summary = (
    features[TARGET_COLUMN]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

target_summary["pct"] = (
    target_summary["count"]
    / len(features)
    * 100
)

target_summary.index = target_summary.index.map(
    {
        0: "on_time",
        1: "late",
    }
)

display(target_summary)

assert int(target_summary.loc["on_time", "count"]) == 88_627
assert int(target_summary.loc["late", "count"]) == 7_823

,count,pct
late_delivery,,
on_time,88627,91.8891
late,7823,8.1109


## 11. Chronological train / validation / test split

A random split is intentionally avoided.

The split is ordered by `prediction_timestamp`:

- Train: first 70%
- Validation: next 15%
- Test: final 15%

This better simulates deployment on future orders and reduces temporal leakage.

In [12]:
split = make_chronological_splits(
    features,
    train_fraction=0.70,
    validation_fraction=0.15,
)

train_df = split.train
validation_df = split.validation
test_df = split.test

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "start": train_df["prediction_timestamp"].min(),
            "end": train_df["prediction_timestamp"].max(),
            "late_rate": train_df[TARGET_COLUMN].mean(),
        },
        {
            "split": "validation",
            "rows": len(validation_df),
            "start": validation_df["prediction_timestamp"].min(),
            "end": validation_df["prediction_timestamp"].max(),
            "late_rate": validation_df[TARGET_COLUMN].mean(),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "start": test_df["prediction_timestamp"].min(),
            "end": test_df["prediction_timestamp"].max(),
            "late_rate": test_df[TARGET_COLUMN].mean(),
        },
    ]
)

display(split_summary)

,split,rows,start,end,late_rate
0,train,67515,2016-09-15 12:16:38,2018-04-16 08:51:19,0.0902
1,validation,14467,2018-04-16 08:51:20,2018-06-21 11:59:03,0.0538
2,test,14468,2018-06-21 12:21:56,2018-08-29 15:10:26,0.0658


### 11.1 Split invariants

In [13]:
assert (
    train_df["prediction_timestamp"].max()
    <= validation_df["prediction_timestamp"].min()
)

assert (
    validation_df["prediction_timestamp"].max()
    <= test_df["prediction_timestamp"].min()
)

assert set(train_df["order_id"]).isdisjoint(
    validation_df["order_id"]
)
assert set(train_df["order_id"]).isdisjoint(
    test_df["order_id"]
)
assert set(validation_df["order_id"]).isdisjoint(
    test_df["order_id"]
)

print("✓ Chronological split invariants passed.")

✓ Chronological split invariants passed.


## 12. Preprocessing contract

Install scikit-learn if it is not already in the project:

`uv add scikit-learn`

The reusable preprocessing implementation is located at:

`src/features/preprocessing.py`

Important: preprocessing must be fitted on **train only**. Validation and test data must only be transformed with the already fitted pipeline.

In [14]:
try:
    from src.features.preprocessing import (
        build_preprocessor,
        select_model_matrix,
    )

    preprocessor = build_preprocessor()

    X_train = select_model_matrix(train_df)
    y_train = train_df[TARGET_COLUMN].copy()

    X_validation = select_model_matrix(validation_df)
    y_validation = validation_df[TARGET_COLUMN].copy()

    X_test = select_model_matrix(test_df)
    y_test = test_df[TARGET_COLUMN].copy()

    print("X_train:", X_train.shape)
    print("X_validation:", X_validation.shape)
    print("X_test:", X_test.shape)
    print("✓ Preprocessing contract loaded.")
except ModuleNotFoundError as exc:
    print(
        "scikit-learn is not installed yet. "
        "Run: uv add scikit-learn"
    )
    print("Details:", exc)

X_train: (67515, 41)
X_validation: (14467, 41)
X_test: (14468, 41)
✓ Preprocessing contract loaded.


## 13. Export processed datasets

We export the **untransformed** feature datasets.

Do not export a globally fitted/scaled matrix because that would make it easy to introduce preprocessing leakage.

Phase 4 will combine the preprocessor and estimator inside an sklearn `Pipeline`.

In [15]:
features.to_parquet(
    PROCESSED_DIR / "features.parquet",
    index=False,
)

train_df.to_parquet(
    PROCESSED_DIR / "train.parquet",
    index=False,
)

validation_df.to_parquet(
    PROCESSED_DIR / "validation.parquet",
    index=False,
)

test_df.to_parquet(
    PROCESSED_DIR / "test.parquet",
    index=False,
)

feature_contract.to_csv(
    METRICS_DIR / "feature_contract.csv",
    index=False,
)

missing_summary.to_csv(
    METRICS_DIR / "feature_missingness.csv",
)

split_summary.to_csv(
    METRICS_DIR / "feature_split_summary.csv",
    index=False,
)

print("✓ Processed feature datasets exported.")

✓ Processed feature datasets exported.


## 14. Final validation

In [16]:
reloaded = pd.read_parquet(
    PROCESSED_DIR / "features.parquet"
)

validate_feature_dataset(
    reloaded,
    expected_order_count=len(orders_cleaned),
)

assert reloaded["order_id"].is_unique
assert len(reloaded) == 96_450

print(
    f"✓ Final processed dataset validated: "
    f"{len(reloaded):,} unique orders."
)

✓ Final processed dataset validated: 96,450 unique orders.


# 15. Conclusions

## Dataset design

The processed feature dataset preserves the modeling grain:

**1 row = 1 order**

The finalized population contains **96,450 orders**.

The target remains:

- On-time: **88,627**
- Late: **7,823**

## Prediction point

The model is designed to make a prediction at:

`order_approved_at`

All model features must be available at or before this timestamp.

## Feature groups

The Phase 3 feature contract includes:

- temporal context;
- approval latency;
- promised delivery window;
- customer geography;
- order-item aggregates;
- price and freight aggregates;
- payment aggregates;
- product weight and volume;
- product category;
- seller geographic diversity;
- seller–customer distance;
- same-state fulfillment.

## Leakage controls

Outcome-derived, post-prediction and review information is explicitly forbidden from the model feature matrix.

The processed dataset intentionally excludes:

- actual customer delivery timestamp;
- carrier delivery timestamp;
- delivery delay days;
- order status;
- review information.

## Missing values

Missing values are retained in the raw feature dataset and will be handled inside the sklearn preprocessing pipeline.

The preprocessing pipeline must be fitted on the training split only.

## Validation strategy

The dataset is split chronologically rather than randomly:

- 70% train;
- 15% validation;
- 15% test.

The test set therefore represents the latest orders and better approximates future deployment.

## Outputs

Phase 3 produces:

- `data/processed/features.parquet`
- `data/processed/train.parquet`
- `data/processed/validation.parquet`
- `data/processed/test.parquet`
- `reports/metrics/feature_contract.csv`
- `reports/metrics/feature_missingness.csv`
- `reports/metrics/feature_split_summary.csv`

## Next step

Proceed to **Phase 4 — Model Development & Experimentation**.

Start with:

- DummyClassifier;
- Logistic Regression;
- Decision Tree;
- Random Forest;
- gradient-boosted tree model.

Use PR-AUC as the primary comparison metric and track Recall, Precision, F1 and ROC-AUC.